In [0]:
%pip install langchain
dbutils.library.restartPython()

  Obtaining dependency information for langchain from https://files.pythonhosted.org/packages/51/3f/462c134228fbb4f65be0a9db6a651e2f1d7226d003a712f1bac455a141b7/langchain-0.3.1-py3-none-any.whl.metadata
  Obtaining dependency information for SQLAlchemy<3,>=1.4 from https://files.pythonhosted.org/packages/8c/d6/97bdc8d714fb21762f2092511f380f18cdb2d985d516071fa925bb433a90/SQLAlchemy-2.0.35-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for aiohttp<4.0.0,>=3.8.3 from https://files.pythonhosted.org/packages/54/76/b106eb516d327527a6b1e0409a3553745ad34480eddfd0d7cad48ddc9848/aiohttp-3.10.8-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for langchain-core<0.4.0,>=0.3.6 from https://files.pythonhosted.org/packages/59/5a/24d328d741d94d6580b228e26e0c949d3e1c2e613c4a26d98ddce1182093/langchain_core-0.3.7-py3-none-any.whl.metadata
  Obtaining dependency information for langchain-text-spli

In [0]:
import os
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Directory path containing multiple CSV files
directory_path = "/Volumes/databricks_hackathon/llm/rag/csv"

# List all CSV files in the directory
file_paths = [file.path for file in dbutils.fs.ls(directory_path) if file.path.endswith('.csv')]

# Function to process a single file and split its text into chunks
def process_file(file_path):
    # Read the text file
    df = spark.read.text(file_path)
    
    # Collect all the text into a single string
    text_column = " ".join([row.value for row in df.collect()])
    
    # Initialize the text splitter
    splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", " ", ""],
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
    )
    
    # Split the text into chunks
    chunks = splitter.split_text(text_column)
    
    return chunks

# Loop through all the CSV files and process them
for file_path in file_paths:
    chunks = process_file(file_path)
    
    # Perform actions with the chunks (e.g., passing them to the next notebook)
    # This could involve saving them to a temporary location, using them in memory, etc.
    for i, chunk in enumerate(chunks):
        print(f"Processing {file_path} - Chunk {i + 1}:\n{chunk}\n")

# Now the chunks are processed and can be used for the next steps in the Databricks workflow


Processing dbfs:/Volumes/databricks_hackathon/llm/rag/csv/anz_physcial_asset.csv - Chunk 1:
MDM ID Property,MDM ID Deal,Country,Fund,Investment Deal,Property Name,Sector,Sub-Sector,Asset Subtype,Property Address,State,City,Zip Code,Portfolio Company,Total Square Feet,Total Square Meters,JV / Operating Partner,JV Partner,Operating Partner,Property Manager,Acquisition Date,Year Built ,Latitude,Longitude,Investment Description,Insured Name,Entity Name,Street Address,Building Description,Construction Overview,Roof Construction (Category),Details of the Roof Construction,"Wall Construction (Category)",Details of the Wall Construction,Details of the Floor Construction,Main Tenant(s),"Main Business Activity of Tenant(s) (Category)",Details of Main Business Activity of Tenant(s),Products Stored or Manufactured,Details of the Fire Protection Overview,Sprinklers (Yes/No),Fire Hose / Fire Extinguisher (Yes/No),Security Overview,CCTV (Yes/No),Security Guards / Security Alarm / Security Gate & Fenc

In [0]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, StringType
import pandas as pd

@pandas_udf("array<string>")
def get_chunks(dummy):
    return pd.Series([chunks])

# Register the UDF
spark.udf.register("get_chunks_udf", get_chunks)

In [0]:
%sql
insert into databricks_hackathon.llm.docs_text (text)
select explode(get_chunks_udf('dummy')) as text;

num_affected_rows,num_inserted_rows
19,19


In [0]:
# Assuming you have a CSV file you want to read into a DataFrame
df = spark.read.csv("/path/to/your/data.csv", header=True, inferSchema=True)

# Now you can create a temporary view from the DataFrame
df.createOrReplaceTempView("temp_table")

# Your SQL operation
spark.sql("""
    INSERT INTO databricks_hackathon.llm.docs_track
    SELECT * FROM temp_table
    WHERE NOT EXISTS (
        SELECT 1 FROM databricks_hackathon.llm.docs_track
        WHERE temp_table.file_name = databricks_hackathon.llm.docs_track.file_name
    )
""")


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4369964190003908>, line 2
      1 # Create a temporary table from the DataFrame
----> 2 df.createOrReplaceTempView("temp_table")
      4 # Ensure the catalog, schema, and table names are correct
      5 # Adjust the query to use the correct names if they were misspelled
      6 spark.sql("""
      7     INSERT INTO databricks_hackathon.llm.docs_track
      8     SELECT temp_table.*
   (...)
     11     ON temp_table.value = databricks_hackathon.llm.docs_track.file_name
     12 """)

NameError: name 'df' is not defined